## 04a - Model selection

Trains BERTopic with several values of the HDBSCAN `min_cluster_size` and of the CountVectorizer `min_df`, and compares
them to check the values used in 04b. No model is saved.

Input: `data/processed/trenord_reviews_clean.csv` (Italian reviews) and `data/processed/trenord_reviews_topics.csv` (for the check with 30)

Output: tables of metrics in this notebook

In [1]:
# --- IMPORTS ---
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords
from umap import UMAP

# Values to compare. 04b uses min_cluster_size=30 and min_df=5.
MIN_CLUSTER_SIZES = [20, 25, 30, 40, 50]
MIN_DF_VALUES = [1, 2, 5]

W1102 18:38:26.120000 29576 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# --- LOAD DATA AND PREPARE TEXTS ---
# Same filter, row order and text column as 04b, so the run with 30 can be compared with its output.
df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
italian_df = df[df["language"] == "it"].copy()

texts = italian_df["text_clean"].tolist()

# BERTopic cannot handle empty texts, so they become a single space, as in 04b
for i in range(len(texts)):
    text = texts[i]
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

print(f"Italian reviews: {len(texts)}")

Italian reviews: 6951


In [3]:
# --- SENTENCE EMBEDDINGS (computed once) ---
# Encoding is the slowest step and does not depend on the parameters compared here,
# so every run below reuses these vectors.
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
embeddings = embedding_model.encode(texts, show_progress_bar=False)

print(f"Embeddings: {embeddings.shape[0]} reviews x {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 14/199 [00:00<00:01, 137.61it/s]

Loading weights:  14%|█▍        | 28/199 [00:00<00:02, 72.08it/s] 

Loading weights:  19%|█▊        | 37/199 [00:00<00:03, 43.11it/s]

Loading weights:  29%|██▉       | 58/199 [00:01<00:02, 49.67it/s]

Loading weights:  32%|███▏      | 64/199 [00:01<00:03, 44.18it/s]

... (2 outputs left out)

Loading weights:  54%|█████▍    | 107/199 [00:01<00:00, 95.24it/s]

Loading weights:  60%|█████▉    | 119/199 [00:01<00:01, 73.18it/s]

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 102.34it/s]

Embeddings: 6951 reviews x 768 dimensions


In [4]:
# --- TOKENIZED CORPUS FOR COHERENCE ---
# Coherence checks how often the top words of a topic appear together in the reviews.
# The reviews are split with the same analyzer that produces the topic keywords
# (Italian stopwords removed, single words and bigrams), so that a bigram keyword
# like "acquistare biglietti" can be found in the corpus.
coherence_vectorizer = CountVectorizer(stop_words=list(italian_stopwords), ngram_range=(1, 2))
analyzer = coherence_vectorizer.build_analyzer()

tokenized_texts = []
for text in texts:
    tokenized_texts.append(analyzer(text))

dictionary = Dictionary(tokenized_texts)

print(f"Tokenized reviews: {len(tokenized_texts)}")
print(f"Distinct tokens  : {len(dictionary)}")

Tokenized reviews: 6951
Distinct tokens  : 52520


In [5]:
# --- FUNCTIONS USED BY BOTH COMPARISONS ---
# Both comparisons train the same pipeline and compute the same metrics,
# so the steps are written once here.

def train_topic_model(min_cluster_size, min_df):
    # Everything except min_cluster_size and min_df is the same as in 04b. Each run gets
    # fresh model objects, because BERTopic fits them in place.
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean", prediction_data=True)
    vectorizer_model = CountVectorizer(stop_words=list(italian_stopwords), min_df=min_df, ngram_range=(1, 2))
    representation_model = MaximalMarginalRelevance(diversity=0.3)

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        top_n_words=10,
        language="multilingual",
        calculate_probabilities=False,
        verbose=False
    )

    # Passing the embeddings skips the encoding step inside BERTopic
    topics_before, _ = topic_model.fit_transform(texts, embeddings=embeddings)

    # Outliers are reassigned to the closest topic, as in 04b
    topics_after = topic_model.reduce_outliers(
        texts,
        topics_before,
        strategy="embeddings",
        threshold=0.0,
        embeddings=embeddings
    )
    topic_model.update_topics(
        texts,
        topics=topics_after,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model
    )
    return topic_model, topics_before, topics_after


def compute_metrics(topic_model, topics_before, topics_after):
    topic_ids = []
    for topic_id in topic_model.get_topics():
        if topic_id != -1:
            topic_ids.append(topic_id)

    # Share of reviews that HDBSCAN left without a topic, before they were reassigned
    outlier_share = topics_before.count(-1) / len(topics_before) * 100

    # Share of the largest topic after reduction: shows when one topic swallows most reviews
    largest_topic_size = pd.Series(topics_after).value_counts().max()
    largest_topic_share = largest_topic_size / len(topics_after) * 100

    # Top 10 words of each topic. BERTopic pads a topic with empty strings when it has
    # fewer words, and gensim stops on a word that is not in the dictionary,
    # so both are counted and left out of the coherence.
    words_for_coherence = []
    all_top_words = []
    empty_words = 0
    words_not_found = 0
    for topic_id in topic_ids:
        topic_words = []
        for word, _ in topic_model.get_topic(topic_id)[:10]:
            if word == "":
                empty_words += 1
                continue
            all_top_words.append(word)
            if word not in dictionary.token2id:
                words_not_found += 1
                continue
            topic_words.append(word)
        words_for_coherence.append(topic_words)

    # processes=1 avoids starting worker processes from the notebook on Windows
    coherence_model = CoherenceModel(
        topics=words_for_coherence,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence="c_v",
        processes=1
    )
    coherence = coherence_model.get_coherence()

    # 1.0 would mean that no word is shared between the top words of two topics
    diversity = len(set(all_top_words)) / len(all_top_words)

    return {
        "topics": len(topic_ids),
        "outliers before reduction (%)": round(outlier_share, 1),
        "largest topic (%)": round(largest_topic_share, 1),
        "coherence c_v": round(coherence, 3),
        "diversity": round(diversity, 3),
        "topic words not found": words_not_found,
        "empty topic words": empty_words,
    }

### min_cluster_size

Smallest group of reviews that HDBSCAN accepts as a topic. `min_df` stays at 5, as in 04b.

In [6]:
# --- TRAIN ONE MODEL PER min_cluster_size ---
runs = {}

for min_cluster_size in MIN_CLUSTER_SIZES:
    topic_model, topics_before, topics_after = train_topic_model(min_cluster_size, min_df=5)
    runs[min_cluster_size] = {
        "model": topic_model,
        "topics_before": topics_before,
        "topics_after": topics_after,
    }
    print(f"min_cluster_size={min_cluster_size}: trained")

2026-11-02 18:46:30,356 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_cluster_size=20: trained


2026-11-02 18:47:44,960 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_cluster_size=25: trained


2026-11-02 18:48:46,981 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_cluster_size=30: trained


2026-11-02 18:49:19,714 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_cluster_size=40: trained


2026-11-02 18:49:53,252 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_cluster_size=50: trained


In [7]:
# --- METRICS FOR EACH min_cluster_size ---
rows = []

for min_cluster_size in MIN_CLUSTER_SIZES:
    run = runs[min_cluster_size]
    metrics = compute_metrics(run["model"], run["topics_before"], run["topics_after"])
    row = {"min_cluster_size": min_cluster_size}
    row.update(metrics)
    rows.append(row)

results = pd.DataFrame(rows).set_index("min_cluster_size")
results

,topics,outliers before reduction (%),largest topic (%),coherence c_v,diversity,topic words not found,empty topic words
min_cluster_size,,,,,,,
20,55,26.5,14.8,0.416,0.703,0,5
25,37,17.8,28.6,0.421,0.684,0,0
30,30,23.0,17.5,0.419,0.783,0,0
40,8,1.7,85.9,0.348,0.212,0,14
50,6,1.8,85.9,0.497,0.192,0,34


In [8]:
# --- REPRODUCIBILITY CHECK FOR min_cluster_size=30 ---
# The run with 30 should give back the model trained in 04b: same topics with the same sizes.
saved_topics = pd.read_csv("../data/processed/trenord_reviews_topics.csv")["topic"].tolist()
run_topics = list(runs[30]["topics_after"])

saved_sizes = pd.Series(saved_topics).value_counts().sort_index()
run_sizes = pd.Series(run_topics).value_counts().sort_index()

checks = {
    "same number of topics": len(saved_sizes) == len(run_sizes),
    "same size per topic": saved_sizes.equals(run_sizes),
    "same topic for every review": saved_topics == run_topics,
}

print(f"Topics in topics.csv / in this run: {len(saved_sizes)} / {len(run_sizes)}")
for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

Topics in topics.csv / in this run: 30 / 30
PASS  same number of topics
PASS  same size per topic
PASS  same topic for every review


### Keyword filter: min_df

In BERTopic the vectorizer is fitted on one joined document per topic, so `min_df` counts topics, not reviews, and it
changes only the keywords of each topic, not which reviews belong to which topic. The runs below use `min_cluster_size=30`.

In [9]:
# --- TRAIN ONE MODEL PER min_df (min_cluster_size=30) ---
min_df_runs = {}

for min_df in MIN_DF_VALUES:
    topic_model, topics_before, topics_after = train_topic_model(30, min_df=min_df)
    min_df_runs[min_df] = {
        "model": topic_model,
        "topics_before": topics_before,
        "topics_after": topics_after,
    }
    print(f"min_df={min_df}: trained")

2026-11-02 18:50:44,567 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_df=1: trained


2026-11-02 18:51:46,406 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_df=2: trained


2026-11-02 18:52:42,729 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


min_df=5: trained


In [10]:
# --- SAME TOPIC ASSIGNMENTS FOR EVERY min_df ---
# min_df only filters the vocabulary used for keywords, so every review should keep its topic.
reference_topics = list(min_df_runs[5]["topics_after"])

for min_df in MIN_DF_VALUES:
    same_topics = list(min_df_runs[min_df]["topics_after"]) == reference_topics
    print(f"{'PASS' if same_topics else 'FAIL'}  min_df={min_df}: same topic for every review as min_df=5")

same_as_first_comparison = reference_topics == list(runs[30]["topics_after"])
print(f"{'PASS' if same_as_first_comparison else 'FAIL'}  min_df=5: same topic for every review as min_cluster_size=30 above")

PASS  min_df=1: same topic for every review as min_df=5
PASS  min_df=2: same topic for every review as min_df=5
PASS  min_df=5: same topic for every review as min_df=5
PASS  min_df=5: same topic for every review as min_cluster_size=30 above


In [11]:
# --- METRICS FOR EACH min_df ---
min_df_rows = []

for min_df in MIN_DF_VALUES:
    run = min_df_runs[min_df]
    metrics = compute_metrics(run["model"], run["topics_before"], run["topics_after"])
    min_df_rows.append({
        "min_df": min_df,
        "coherence c_v": metrics["coherence c_v"],
        "diversity": metrics["diversity"],
        "empty topic words": metrics["empty topic words"],
        "topic words not found": metrics["topic words not found"],
    })

min_df_results = pd.DataFrame(min_df_rows).set_index("min_df")
min_df_results

,coherence c_v,diversity,empty topic words,topic words not found
min_df,,,,
1,0.385,0.887,0,49
2,0.409,0.863,0,20
5,0.419,0.783,0,0


In [12]:
# --- TOP 5 KEYWORDS OF FIVE TOPICS, SIDE BY SIDE ---
# Topic ids are the same for every min_df, because the assignments are identical.
# These topics were chosen so that none of the keywords shown is a personal name or a code.
TOPICS_TO_SHOW = [5, 9, 10, 14, 26]

# Show the full keyword lists instead of cutting them at 50 characters
pd.set_option("display.max_colwidth", None)

keyword_rows = []
for topic_id in TOPICS_TO_SHOW:
    row = {"topic": topic_id}
    for min_df in MIN_DF_VALUES:
        words = []
        for word, _ in min_df_runs[min_df]["model"].get_topic(topic_id)[:5]:
            words.append(word)
        row[f"min_df={min_df}"] = ", ".join(words)
    keyword_rows.append(row)

keywords_table = pd.DataFrame(keyword_rows).set_index("topic")
keywords_table

,min_df=1,min_df=2,min_df=5
topic,,,
5,"password, login, account, app, errore","password, login, account, accesso, errore","login, account, accesso, errore, mail"
9,"utile utile, ottima applicazione, utile funzionale, funzionale ottima, utilissima","funzionale, ottima applicazione, utile funzionale, funzionale ottima, applicazione utile","ottima applicazione, applicazione, utilissima, applicazione utile, efficace"
10,"ottima ottima, perfetto ottima, ottimale ottima, bellissimo perfetta, spettacolare ottima","ottima ottima, ottima, fantastica ottima, ottimale, ottimo sevizio","ottima, perfetta, perfetto, fantastica, bellissima"
14,"ottimo ottimo, ottimo ok, buono ottimo, servizio ottimo, ottimo buona","ottimo servizio, ottima esperienza, buon servizio, complimenti ottimo, ottimo supporto","ottimo, ottimo servizio, buono, complimenti, servizio"
26,"funziona funziona, accedere funziona, funziona impossibile, comando disconnettersi, bug funziona","funziona impossibile, utilizzare tap, accesso funziona, risolvendo problema, settimana funziona","funziona, accedere, assistenza, accesso, impossibile utilizzare"


### How to read the tables

min_cluster_size

- Topics: 20 gives 55 topics, 25 gives 37 and 30 gives 30. At 40 and 50 the count falls to 8 and 6, because most reviews end up in one topic.
- Outliers before reduction: 25 leaves the fewest among the usable values (17.8%), 30 leaves 23.0%. All outliers are reassigned afterwards, so this mostly says how much of each topic comes from the reassignment.
- Largest topic: with 30 the largest topic holds 17.5% of the reviews, with 25 it holds 28.6%, with 40 and 50 it holds 85.9%.
- Coherence c_v: 20, 25 and 30 are within 0.005 of each other. It is a proxy and also rewards near-synonyms, so the 0.497 of 50 comes from a few generic words shared by a handful of topics, not from better topics.
- Diversity: 30 has the highest value (0.783), 20 and 25 repeat more words between topics.
- Empty topic words: none for 25 and 30, 5 for 20, 14 and 34 for 40 and 50.

30 is the best trade-off: coherence as high as 20 and 25, the highest diversity, no empty keyword slots and no topic above a fifth of the reviews. 25 is the closest alternative, with fewer outliers but one topic twice as large and more repeated keywords. 40 and 50 collapse into one big topic.

min_df

- Assignments: the three values give the same topic for every review, and min_df=5 matches the run above.
- Topic words not found: with min_df 1 and 2, 49 and 20 keywords never appear in a single review. The reviews of a topic are joined into one text, so a bigram can join the last word of one review with the first word of the next, like "ottima ottima" or "perfetto ottima". A pair like this rarely shows up in five topics, so min_df=5 drops it.
- Coherence c_v: 0.385, 0.409 and 0.419. It rises with min_df even though the words not found are left out of the score for 1 and 2.
- Diversity: higher with lower min_df (0.887 for 1), but the gain comes from rare variants and joined pairs that count as different strings, as the keyword table shows.
- Empty topic words: none for all three with 30 topics. Empty slots appear only when there are few topics, as with 40 and 50, because a word then has to appear in 5 of only 6 or 8 topics.

min_df=5 is the best choice here: the keywords are words that really occur in reviews and coherence is the highest. The cost is that a word found in fewer than five topics is lost even when it is the most specific one, like "password" for topic 5. If the number of topics is reduced a lot, min_df should go down with it.

The metrics do not replace reading the topics. The labels in 04c come from reading the reviews of each topic.